In [3]:
import os, sys, shutil, subprocess
from multiprocessing import cpu_count

print("Python:", sys.version)
print("CPU cores:", cpu_count())

# Kaggle paden
BASE = "/kaggle/working"
APPLIO_DIR = f"{BASE}/Applio"
LOGS_DIR = f"{APPLIO_DIR}/logs"

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CPU cores: 4


In [4]:
APPLIO_REF = "3.6.2"   # probeer bv "main" of een release-tag als dat beter werkt

%cd /kaggle/working
if not os.path.isdir(APPLIO_DIR):
    !git clone https://github.com/IAHispano/Applio --branch {APPLIO_REF} --single-branch

# System deps (portaudio is vaak nodig)
!apt-get update -y
!apt-get install -y portaudio19-dev psmisc

%cd {APPLIO_DIR}

# Snelle/strakke pip via uv (zoals Applio notebooks doen)
!pip -q install uv
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system

# Download Applio prerequisites (models/pretraineds etc.)
!python core.py prerequisites --models "True" --pretraineds_hifigan "True" > /dev/null 2>&1

print("✅ Install klaar")

/kaggle/working
Cloning into 'Applio'...
remote: Enumerating objects: 19967, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 19967 (delta 84), reused 72 (delta 60), pack-reused 19846 (from 3)
Receiving objects: 100% (19967/19967), 49.28 MiB | 37.30 MiB/s, done.
Resolving deltas: 100% (13064/13064), done.
Note: switching to 'c7be7282af0f7b6382cbb2ac780b11242da12916'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Get:1 https://cli.github.com/packages stable 

In [5]:
!python /kaggle/working/Applio/core.py preprocess \
  --model_name "hitsugi_rvc_v2" \
  --dataset_path "/kaggle/input/datasets/michaeltje26/hitsugi" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --cut_preprocess "Automatic" \
  --process_effects "False" \
  --noise_reduction "False" \
  --chunk_len "3" \
  --overlap_len "0.3" \
  --normalization_mode "none"

Starting preprocess with 4 processes...
100%|█████████████████████████████████████████████| 4/4 [00:32<00:00,  8.01s/it]
Preprocess completed in 32.06 seconds on 00:34:24 seconds of audio.


In [6]:
!python /kaggle/working/Applio/core.py extract \
  --model_name "hitsugi_rvc_v2" \
  --f0_method "rmvpe" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --gpu "0" \
  --embedder_model "contentvec" \
  --embedder_model_custom "" \
  --include_mutes "2"

Starting pitch extraction on cuda:0 using rmvpe...
100%|█████████████████████████████████████████| 645/645 [00:16<00:00, 38.56it/s]
Pitch extraction completed in 24.35 seconds.
Starting embedding extraction with 4 cores on cuda:0...
100%|█████████████████████████████████████████| 645/645 [00:11<00:00, 58.10it/s]
Embedding extraction completed in 17.81 seconds.


In [7]:
!python /kaggle/working/Applio/core.py index \
  --model_name "hitsugi_rvc_v2" \
  --index_algorithm "Auto"

Saved index file '/kaggle/working/Applio/logs/hitsugi_rvc_v2/hitsugi_rvc_v2.index'


In [8]:
!python /kaggle/working/Applio/core.py train \
  --model_name "hitsugi_rvc_v2" \
  --save_every_epoch "5" \
  --save_only_latest "True" \
  --save_every_weights "False" \
  --total_epoch "200" \
  --sample_rate "40000" \
  --batch_size "8" \
  --gpu 0 \
  --pretrained "True" \
  --custom_pretrained "False" \
  --overtraining_detector "False" \
  --cleanup "False" \
  --cache_data_in_gpu "False" \
  --vocoder "HiFi-GAN" \
  --checkpointing "False"

2026-03-02 17:47:24.805677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772473645.266119    1178 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772473645.386254    1178 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772473646.401754    1178 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772473646.401830    1178 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772473646.401834    1178 computation_placer.cc:177] computation placer alr

In [15]:
import os

paths = [
    "/kaggle/working/hitsugi_rvc_v2_inference.zip",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/wavs",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/raw",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/f0",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/features",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/tensorboard"
]

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            import shutil
            shutil.rmtree(p)
        else:
            os.remove(p)
        print("Deleted:", p)


In [12]:
import torch, json
from pathlib import Path

RUN_DIR = Path("/kaggle/working/Applio/logs/hitsugi_rvc_v2")

# pak nieuwste G
g_list = sorted(RUN_DIR.glob("G_*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
G_PATH = g_list[0]

cfg = json.loads((RUN_DIR / "config.json").read_text())

export = {
    "weight": torch.load(G_PATH, map_location="cpu")["model"],
    "config": {
        "sr": cfg.get("sr", 40000),
        "f0": cfg.get("f0", 1),
        "version": cfg.get("version", "v2")
    }
}

OUT = RUN_DIR / "hitsugi_rvc_v2_export.pth"
torch.save(export, OUT)

print("Exported:", OUT)


Exported: /kaggle/working/Applio/logs/hitsugi_rvc_v2/hitsugi_rvc_v2_export.pth


In [27]:
%%bash
cd /kaggle/working
cp hitsugi_rvc_v2_export.pth hitsugi_rvc_v2_export.bin
cp hitsugi_rvc_v2_export.pth hitsugi_rvc_v2_export.txt
ls -lh hitsugi_rvc_v2_export.*

-rw-r--r-- 1 root root 140M Mar  2 21:35 hitsugi_rvc_v2_export.bin
-rw-r--r-- 1 root root 140M Mar  2 20:59 hitsugi_rvc_v2_export.pth
-rw-r--r-- 1 root root 140M Mar  2 21:35 hitsugi_rvc_v2_export.txt
